In [1]:
import pickle

# run the model
with open('../xgb_model_withgpa.pkl', 'rb') as file:
    xgb_model = pickle.load(file)



In [2]:
import pandas as pd
import numpy as np

In [3]:
xgb_input = pd.read_csv("../x.csv")
xgb_input.head(10)

,Unnamed: 0,anxious,calm,conventional,critical,dependable,disorganized,enthusiastic,experiences,reserved,...,hour,exercise,walk,have,schedule,sleep_hours,sleep_quality,sleepiness,number,gpa_all
0,0,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
1,1,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
2,2,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
3,3,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
4,4,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
5,5,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
6,6,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
7,7,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
8,8,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
9,9,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505


## Noise experiment on highly relevent variables

In [4]:
print("anxious distinct values:", xgb_input['anxious'].unique())
print("sleep_hours distinct values:", xgb_input['sleep_hours'].unique())
print("gpa_all distinct values:", xgb_input['gpa_all'].unique())


anxious distinct values: [1. 2. 4. 3. 5.]
sleep_hours distinct values: [ 8.  7.  9.  6. 12.  5. 10.  0.  1. 15. 11.  4. 13.  3.  2. 14. 16. 19.]
gpa_all distinct values: [3.505 3.029 3.474 3.705 3.667 3.245 3.293 3.373 3.476 3.947 3.719 3.826
 2.815 3.79  3.625 2.4   3.519]


In [5]:
# add noise to variables: anxious
# anxious is ordinal variable
# Randomly add 1, subtract 1, or make no change for each sample.

noisy_input = xgb_input.copy()
anxious_noise = np.random.choice([-1, 0, 1], size=noisy_input.shape[0])
noisy_input['anxious'] = noisy_input['anxious'] + anxious_noise
noisy_input['anxious'] = noisy_input['anxious'].clip(lower=1, upper=5)
print(noisy_input['anxious'].value_counts().sort_index())


anxious
1.0    3221
2.0    2770
3.0    1951
4.0     734
5.0     327
Name: count, dtype: int64


In [6]:
# add noise to variable: sleep_hours
# sleep_hours is discrete quantitative variable
# apply normal distribution to choose the reasonable noise to add on the original sleep_hours column

sleep_noise = np.random.normal(0, 1, size=noisy_input.shape[0]) 
noisy_input['sleep_hours'] = noisy_input['sleep_hours'] + sleep_noise
noisy_input['sleep_hours'] = noisy_input['sleep_hours'].round().astype(int)
noisy_input['sleep_hours'] = noisy_input['sleep_hours'].clip(lower=0, upper=24)
print(noisy_input['sleep_hours'].head(10))


0     9
1     9
2     7
3     7
4     8
5     6
6     9
7    11
8     8
9     8
Name: sleep_hours, dtype: int64


In [7]:
# add noise to variable: gpa_all
# gpa_all is continuous numerical variable
# apply normal distribution to choose the reasonable noise to add on the original sleep_hours column

gpa_noise = np.random.normal(0, 0.05, size=noisy_input.shape[0])
noisy_input['gpa_all'] = noisy_input['gpa_all'] + gpa_noise
print(noisy_input['gpa_all'].head(10))

0    3.542260
1    3.467054
2    3.540270
3    3.477232
4    3.520981
5    3.510676
6    3.427868
7    3.507007
8    3.496863
9    3.471065
Name: gpa_all, dtype: float64


In [8]:
# re-run the xgboost model
noisy_input = noisy_input.drop(columns=['Unnamed: 0'], errors='ignore')

y_pred_noisy = xgb_model.predict(noisy_input)


In [9]:
from sklearn.metrics import mean_squared_error
y_data = pd.read_csv("../y_new.csv")

y_true = y_data['total_score']
mse_noisy = mean_squared_error(y_true, y_pred_noisy)

print("MSE after adding noise on highly relevant variables:", mse_noisy)


MSE after adding noise on highly relevant variables: 16.5223677931369


In [10]:
from sklearn.metrics import r2_score
r2_noisy = r2_score(y_true, y_pred_noisy)
print("R² after adding noise on highly relevant variables:", r2_noisy)

R² after adding noise on highly relevant variables: 0.48232658788532


## Noise experiment on less relevent variables

In [11]:
print("conventional distinct values:", xgb_input['conventional'].unique())
print("dependable distinct values:", xgb_input['dependable'].unique())
print("critical distinct values:", xgb_input['critical'].unique())

conventional distinct values: [2. 3. 4. 5. 1.]
dependable distinct values: [2. 3. 4. 1. 5.]
critical distinct values: [2. 3. 4. 1. 5.]


In [12]:
less_relevent_noisy_input = xgb_input.copy()

less_relevent_features_to_noise = ['conventional', 'dependable', 'critical']

for feature in less_relevent_features_to_noise:
    noise = np.random.choice([-1, 0, 1], size=noisy_input.shape[0])
    noisy_input[feature] = noisy_input[feature] + noise
    noisy_input[feature] = noisy_input[feature].clip(lower=1, upper=5)  

print(noisy_input[less_relevent_features_to_noise].head(10))

   conventional  dependable  critical
0           2.0         3.0       1.0
1           2.0         2.0       1.0
2           1.0         2.0       2.0
3           1.0         2.0       1.0
4           1.0         1.0       1.0
5           2.0         1.0       1.0
6           3.0         2.0       3.0
7           2.0         1.0       1.0
8           1.0         3.0       2.0
9           2.0         1.0       2.0


In [13]:
# re-run the xgboost model
less_relevent_noisy_input = noisy_input.drop(columns=['Unnamed: 0'], errors='ignore')

less_relevent_y_pred_noisy = xgb_model.predict(less_relevent_noisy_input)


In [14]:
less_relevent_mse_noisy = mean_squared_error(y_true, less_relevent_y_pred_noisy)

print("MSE after adding noise for less relevant variables:", less_relevent_mse_noisy)

MSE after adding noise for less relevant variables: 16.903935137981374


In [15]:
less_relevent_r2_noisy = r2_score(y_true, less_relevent_y_pred_noisy)
print("R² after adding noise on less relevant variables:", less_relevent_r2_noisy)

R² after adding noise on less relevant variables: 0.47037144490398364


| Condition | MSE | RMSE (√MSE) | R² |
|:---|:---|:---|:---|
| No Noise | 3.3773 | 1.837 | 0.8963 |
| Noise on Highly Relevant Variables | 16.5224 | 4.066 | 0.4823 |
| Noise on Less Relevant Variables | 16.9039 | 4.110 | 0.4704 |
